# Spatial-Temporal Analysis: Collision Hotspots

Finds **where** and **when** collisions concentrate, so road safety officers can see real hotspots on a map.

- **Data:** `master_dataset_v2_engineered.csv`, all 513,801 collisions (2021 to 2025). This task only describes what happened, so there is no train/test split and every row is used.
- **Output:** `hotspots.json` (read by the backend endpoint `/api/hotspots`) and `method_comparison.csv` (evidence for the report).
- **Hotspot subsets:** `all` (every collision) and `severe` (Fatal and Serious only).
- **Time slices:** All times, Night, Morning Rush, Midday, Evening Rush, Evening.

**How to run in Colab**
1. Upload `master_dataset_v2_engineered.csv` to `/content` (left sidebar, Files, Upload).
2. Runtime, Run all. The whole notebook takes about 5 minutes.
3. The last cell downloads the results. Files in `/content` disappear when the session ends.

To check the whole notebook quickly first, set `QUICK_TEST_FRACTION = 0.1` in the first cell. The results from a dry run are not real.

**Steps:** 1 load and check, 2 look at the data, 3 compare four clustering methods and pick one by a stated rule, 4 cluster every subset and time slice, 5 validate and save the hotspot file, 6 read the results.

## 1. Setup, load and check the data

In [ ]:
import json
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN, OPTICS
from sklearn.metrics import silhouette_score, davies_bouldin_score

warnings.filterwarnings("ignore")
SEED = 42

# ---- settings you may change -------------------------------------------------------------
DATA_FILE = Path("/content/master_dataset_v2_engineered.csv")
OUT_DIR = Path("/content/spatial_temporal")
QUICK_TEST_FRACTION = None   # for example 0.1 = dry run on a 10% sample (fast, results are not real)
FORCE_RERUN = False          # True = ignore saved group results and cluster everything again
TOP_N = 1000                 # hotspots kept per subset and time slice
MAX_RADIUS_M = 500           # a hotspot must be compact: 90% of its collisions within this distance of its centre
TILE_M = 25000               # the map is clustered in 25 km tiles (HDBSCAN slows down sharply on huge inputs)
MARGIN_M = 2000              # tiles overlap by 2 km so no hotspot is cut at a tile edge
# ------------------------------------------------------------------------------------------

assert DATA_FILE.exists(), "Upload master_dataset_v2_engineered.csv to /content first (left sidebar > Files)."
OUT_DIR.mkdir(parents=True, exist_ok=True)
GROUP_DIR = OUT_DIR / ("groups_dry_run" if QUICK_TEST_FRACTION else "groups")
GROUP_DIR.mkdir(exist_ok=True)

SLICES = ["All times", "Night", "Morning Rush", "Midday", "Evening Rush", "Evening"]
SEVERITIES = {"all": (1, 2, 3), "severe": (1, 2)}      # severe = Fatal and Serious only

# What counts as a hotspot: (min_cluster_size, min_samples) for the final run.
# A time slice holds only part of the collisions, so its minimum is lowered.
PARAMS = {
    ("all", "All times"): (10, 5),
    ("all", "slice"): (5, 3),
    ("severe", "All times"): (5, 3),
    ("severe", "slice"): (3, 2),
}

In [ ]:
USE = ["collision_index", "latitude", "longitude", "location_easting_osgr", "location_northing_osgr",
       "collision_severity", "time_of_day_bucket", "lsoa_of_accident_location"]
raw = pd.read_csv(DATA_FILE, usecols=USE, dtype={"collision_index": str}, low_memory=False)
assert len(raw) == 513801, f"Expected 513,801 rows, got {len(raw):,}"

COORDS = ["latitude", "longitude", "location_easting_osgr", "location_northing_osgr"]
df = raw.dropna(subset=COORDS).reset_index(drop=True)
assert df.latitude.between(49, 61).all() and df.longitude.between(-9, 2.5).all(), "Coordinates outside Great Britain"
assert set(df.time_of_day_bucket.unique()) == set(SLICES[1:]), "Unexpected time_of_day_bucket values"
assert set(df.collision_severity.unique()) == {1, 2, 3}, "Unexpected severity codes"

print("Rows loaded     :", f"{len(raw):,}")
print("Rows without a location (dropped):", f"{len(raw) - len(df):,}")
print("Rows used       :", f"{len(df):,}")
print("Severity (1=Fatal, 2=Serious, 3=Slight):", df.collision_severity.value_counts().sort_index().to_dict())
print("Time of day     :", df.time_of_day_bucket.value_counts().to_dict())
print("Easting range (m):", int(df.location_easting_osgr.min()), "to", int(df.location_easting_osgr.max()))
print("Northing range (m):", int(df.location_northing_osgr.min()), "to", int(df.location_northing_osgr.max()))

if QUICK_TEST_FRACTION:
    df = df.sample(frac=QUICK_TEST_FRACTION, random_state=SEED).reset_index(drop=True)
    print(f"\nQUICK TEST: using only {len(df):,} rows. Results are NOT real and go to a separate folder.")

## 2. Where do collisions happen?

Clustering uses the **OSGR grid coordinates** (easting and northing in metres), not latitude and longitude. A distance in metres is then exact, and the clustering runs much faster than with haversine distance. Latitude and longitude are still used for the map output.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 7.5))
aspect = 1 / np.cos(np.radians(54))

axes[0].hexbin(df.longitude, df.latitude, gridsize=170, bins="log", mincnt=1, cmap="viridis")
axes[0].set_title("All collisions (colour = count, log scale)")

severe_pts = df[df.collision_severity <= 2]
axes[1].scatter(severe_pts.longitude, severe_pts.latitude, s=0.4, alpha=0.25, color="crimson")
axes[1].set_title("Fatal and Serious collisions")

for ax in axes:
    ax.set_aspect(aspect)
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
plt.tight_layout()
plt.show()

## 3. Which clustering method?

Four methods are compared, as the project plan requires:

| Method | Idea | Why it is here |
|---|---|---|
| K-Means | splits every collision into *k* regions | baseline; cannot mark isolated collisions as noise |
| DBSCAN | groups collisions closer than one fixed radius | classic density method |
| HDBSCAN | like DBSCAN but the radius adapts to local density (finest clusters, `leaf` selection) | cities are dense, countryside is sparse |
| OPTICS | density ordering with a range of radii | strong, but slow |

They are tested on **two 20 km squares of the map**: the densest one (inner London) and a typical one (a few thousand collisions). Both keep the real collision density, which a random sample would not.

**Scores:** silhouette (higher is better) and Davies-Bouldin (lower is better), both on the clustered collisions only, plus the share left as noise, the number of clusters, the typical cluster radius, the largest cluster and the run time. Silhouette alone can mislead: a tiny radius gives a high silhouette while leaving most collisions out. So the choice follows this rule:

1. K-Means is reference only (no noise handling). OPTICS is reference only (too slow for the final runs).
2. A method qualifies only if **at most 50% of collisions are noise in both squares**.
3. Among qualifying methods, the highest **worst-case silhouette** wins.

In [ ]:
def xy(frame):
    return frame[["location_easting_osgr", "location_northing_osgr"]].to_numpy(float)


def window_frames(frame, side=20000):
    # Two 20 km squares: the densest, and a typical one (median count among squares with 2,000 to 6,000 collisions).
    scale = QUICK_TEST_FRACTION or 1
    cx = (frame.location_easting_osgr // side).astype(int)
    cy = (frame.location_northing_osgr // side).astype(int)
    counts = frame.groupby([cx, cy]).size()
    dense_key = counts.idxmax()
    moderate = counts[(counts >= 2000 * scale) & (counts <= 6000 * scale)].sort_values()
    assert len(moderate), "No moderate-density square found"
    typical_key = moderate.index[len(moderate) // 2]
    out = {}
    for name, key in (("Densest square", dense_key), ("Typical square", typical_key)):
        inside = (cx == key[0]) & (cy == key[1])
        out[name] = frame[inside].reset_index(drop=True)
    return out


def describe(P, labels, secs):
    inc = labels >= 0
    k = len(set(labels[inc]))
    out = {"clusters": k, "noise_pct": round(100 * float((~inc).mean()), 1), "secs": round(secs, 1)}
    if k < 2:
        return out
    Q, L = P[inc], labels[inc]
    sizes = np.bincount(L)
    centres = pd.DataFrame(Q).groupby(L).mean()
    dist = np.hypot(Q[:, 0] - centres.loc[L, 0].to_numpy(), Q[:, 1] - centres.loc[L, 1].to_numpy())
    radius = pd.Series(dist).groupby(L).quantile(0.9)
    out.update(
        largest=int(sizes.max()),
        median_radius_m=int(radius.median()),
        silhouette=round(float(silhouette_score(Q, L, sample_size=min(10000, len(Q) - 1), random_state=SEED)), 3),
        davies_bouldin=round(float(davies_bouldin_score(Q, L)), 2),
    )
    return out


def timed(fn):
    t0 = time.time()
    labels = fn()
    return labels, time.time() - t0


windows = window_frames(df)
for name, w in windows.items():
    print(f"{name}: {len(w):,} collisions")

In [ ]:
# DBSCAN needs one radius. Give it its best chance: try several radii in the densest square and keep the one
# with the highest silhouette among those that leave at most 50% of collisions as noise.
P_dense = xy(windows["Densest square"])
grid_rows = []
for eps in (25, 35, 50, 75, 100):
    labels, secs = timed(lambda: DBSCAN(eps=eps, min_samples=10, algorithm="kd_tree").fit_predict(P_dense))
    grid_rows.append({"eps_m": eps, **describe(P_dense, labels, secs)})
dbscan_grid = pd.DataFrame(grid_rows)
print("DBSCAN radius check in the densest square (min_samples = 10):")
print(dbscan_grid.to_string(index=False))

if "silhouette" not in dbscan_grid:
    dbscan_grid["silhouette"] = np.nan
usable = dbscan_grid[(dbscan_grid.noise_pct <= 50) & dbscan_grid.silhouette.notna()]
if len(usable):
    DBSCAN_EPS = int(usable.sort_values("silhouette", ascending=False).iloc[0].eps_m)
else:
    assert QUICK_TEST_FRACTION, "No DBSCAN radius keeps noise at or below 50%. Review the table above."
    DBSCAN_EPS = int(dbscan_grid.sort_values("noise_pct").iloc[0].eps_m)
    print("DRY RUN: no radius met the rule on the small sample; using the one with the least noise.")
print(f"\nDBSCAN radius used in the comparison: {DBSCAN_EPS} m")

In [ ]:
results = []
for wname, frame in windows.items():
    P = xy(frame)
    # HDBSCAN first, so K-Means can be given the same number of clusters.
    lab_h, s_h = timed(lambda: HDBSCAN(min_cluster_size=10, min_samples=5, cluster_selection_method="leaf",
                                        n_jobs=-1).fit_predict(P))
    row_h = describe(P, lab_h, s_h)
    lab_d, s_d = timed(lambda: DBSCAN(eps=DBSCAN_EPS, min_samples=10, algorithm="kd_tree").fit_predict(P))
    k = max(2, row_h["clusters"])
    lab_k, s_k = timed(lambda: KMeans(n_clusters=k, n_init=3, random_state=SEED).fit_predict(P))
    lab_o, s_o = timed(lambda: OPTICS(min_samples=10, max_eps=300, xi=0.05, min_cluster_size=10,
                                      algorithm="kd_tree", n_jobs=-1).fit_predict(P))
    for method, row in (("K-Means", describe(P, lab_k, s_k)), ("DBSCAN", describe(P, lab_d, s_d)),
                        ("HDBSCAN", row_h), ("OPTICS", describe(P, lab_o, s_o))):
        results.append({"window": wname, "collisions": len(P), "method": method, **row})

comparison = pd.DataFrame(results)
for col in ("largest", "median_radius_m", "silhouette", "davies_bouldin"):
    if col not in comparison:
        comparison[col] = np.nan
comparison.to_csv(OUT_DIR / "method_comparison.csv", index=False)
print(comparison.to_string(index=False))

In [ ]:
# Apply the rule from the text above.
ELIGIBLE = ("DBSCAN", "HDBSCAN")
MAX_NOISE = 50

verdict = []
for method in ELIGIBLE:
    r = comparison[comparison.method == method]
    passes = bool((r.noise_pct <= MAX_NOISE).all())
    verdict.append({"method": method, "noise_in_each_square": list(r.noise_pct),
                    "silhouette_in_each_square": list(r.silhouette), "qualifies": passes,
                    "worst_silhouette": float(r.silhouette.min())})
verdict = pd.DataFrame(verdict)
print(verdict.to_string(index=False))

qualified = verdict[verdict.qualifies]
if len(qualified):
    FINAL_METHOD = qualified.sort_values("worst_silhouette", ascending=False).iloc[0].method
else:
    assert QUICK_TEST_FRACTION, "No eligible method keeps noise at or below 50% in both squares. Review the table above."
    FINAL_METHOD = "HDBSCAN"
    print("DRY RUN: no method met the rule on the small sample; using HDBSCAN so the rest of the notebook can be tested.\n")

total_points = 2 * (len(df) + int((df.collision_severity <= 2).sum()))
op = comparison[(comparison.method == "OPTICS") & (comparison.window == "Densest square")].iloc[0]
est_min = op.secs / op.collisions * total_points / 60
print(f"\nFINAL METHOD: {FINAL_METHOD}")
print("K-Means: reference only. It puts every collision in a region and cannot mark isolated collisions as noise.")
print(f"OPTICS : reference only. It took {op.secs:.0f} s for {op.collisions:,} collisions; the final runs cover about "
      f"{total_points:,} collisions, which is at least {est_min:.0f} min (and it slows down faster than linearly).")

## 4. Cluster every subset and time slice

Each of the 12 groups (2 subsets x 6 time slices) is clustered separately with the chosen method. Each group is clustered in 25 km map tiles (with a 2 km overlap, and each cluster credited to the tile holding its centre), because HDBSCAN slows down sharply on very large inputs: in testing, all 513,748 collisions in one piece had not finished after 13 minutes on a single core, while 124,000 took about 75 seconds. The groups run in parallel and each result is saved as it finishes, so if Colab disconnects, running the cell again continues where it stopped.

For every cluster the notebook records the centre (mean latitude and longitude), the radius (the distance containing 90% of its collisions, at least 25 m), the number of collisions by severity, and the most common LSOA code.

**A hotspot must be compact.** Only clusters with a radius of at most `MAX_RADIUS_M` (500 m) are kept, because a hotspot should be a junction or a short stretch of road an officer can act on. Wider clusters are diffuse areas along roads and would mislead as map circles. The `TOP_N` largest compact clusters per group are saved.

In [ ]:
CLUSTER_COLS = ["collisions", "latitude", "longitude", "fatal", "serious", "slight", "radius_m", "lsoa"]


def fit_clusters(method, P, min_cluster_size, min_samples, eps):
    if method == "HDBSCAN":
        return HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples,
                       cluster_selection_method="leaf", n_jobs=1).fit_predict(P)
    if method == "DBSCAN":
        return DBSCAN(eps=eps, min_samples=min_cluster_size, algorithm="kd_tree", n_jobs=1).fit_predict(P)
    raise ValueError(method)


def cluster_table(frame, labels, max_radius, core=None):
    # One row per COMPACT cluster. Label -1 is noise and is ignored. If `core` (x0, x1, y0, y1) is given, only
    # clusters whose centre lies inside that box are returned, so overlapping tiles never count a cluster twice.
    # Returns the table, how many clusters were found, and how many of them were compact.
    f = frame.assign(cluster=labels)
    f = f[f.cluster >= 0]
    if f.empty:
        return pd.DataFrame(columns=CLUSTER_COLS), 0, 0
    f = f.assign(fatal=(f.collision_severity == 1).astype(int),
                 serious=(f.collision_severity == 2).astype(int),
                 slight=(f.collision_severity == 3).astype(int))
    t = f.groupby("cluster").agg(
        collisions=("collision_severity", "size"),
        latitude=("latitude", "mean"), longitude=("longitude", "mean"),
        east=("location_easting_osgr", "mean"), north=("location_northing_osgr", "mean"),
        fatal=("fatal", "sum"), serious=("serious", "sum"), slight=("slight", "sum"))
    if core is not None:
        x0, x1, y0, y1 = core
        t = t[(t.east >= x0) & (t.east < x1) & (t.north >= y0) & (t.north < y1)]
        f = f[f.cluster.isin(t.index)]
    found = len(t)
    if found == 0:
        return pd.DataFrame(columns=CLUSTER_COLS), 0, 0
    f = f.join(t[["east", "north"]], on="cluster")
    dist = np.hypot(f.location_easting_osgr - f.east, f.location_northing_osgr - f.north)
    t["radius_m"] = dist.groupby(f.cluster).quantile(0.9).reindex(t.index).clip(lower=25).round().astype(int)
    t = t[t.radius_m <= max_radius]
    compact = len(t)
    f = f[f.cluster.isin(t.index)]
    known = f[f.lsoa_of_accident_location.notna() & (f.lsoa_of_accident_location.astype(str) != "-1")]
    t["lsoa"] = known.groupby("cluster").lsoa_of_accident_location.agg(lambda s: s.mode().iat[0]).reindex(t.index)
    return t[CLUSTER_COLS].reset_index(drop=True), found, compact


def cluster_group(frame, method, mcs, ms, eps, max_radius):
    # Cluster one group tile by tile. Each tile is clustered with a 2 km margin of neighbouring points,
    # and a cluster is credited to the tile that contains its centre.
    E = frame.location_easting_osgr.to_numpy()
    N = frame.location_northing_osgr.to_numpy()
    cells = pd.DataFrame({"tx": (E // TILE_M).astype(int), "ty": (N // TILE_M).astype(int)})
    sizes = cells.groupby(["tx", "ty"]).size()
    tables, found, compact = [], 0, 0
    for (ix, iy), n_core in sizes.items():
        if n_core < mcs:
            continue                      # too few collisions in this tile to form a cluster
        x0, y0 = ix * TILE_M, iy * TILE_M
        near = ((E >= x0 - MARGIN_M) & (E < x0 + TILE_M + MARGIN_M) &
                (N >= y0 - MARGIN_M) & (N < y0 + TILE_M + MARGIN_M))
        sub = frame[near]
        labels = fit_clusters(method, xy(sub), mcs, ms, eps)
        table, f_, c_ = cluster_table(sub, labels, max_radius, core=(x0, x0 + TILE_M, y0, y0 + TILE_M))
        tables.append(table)
        found += f_
        compact += c_
    tables = [t for t in tables if len(t)]
    out = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame(columns=CLUSTER_COLS)
    out = out.sort_values(["collisions", "fatal", "serious"], ascending=False).reset_index(drop=True)
    return out, found, compact


def group_file(subset, slice_name, method, mcs, ms):
    return GROUP_DIR / f"{subset}__{slice_name.replace(' ', '_')}__{method}_{mcs}_{ms}_r{MAX_RADIUS_M}_t{TILE_M // 1000}_eps{DBSCAN_EPS}.csv"


def run_group(subset, slice_name, frame, method, mcs, ms, eps, keep, max_radius, path, force):
    t0 = time.time()
    if path.exists() and not force:
        saved = pd.read_csv(path)
        return {"subset": subset, "slice": slice_name, "points": int(saved.points.iat[0]) if len(saved) else len(frame),
                "found": int(saved.found.iat[0]) if len(saved) else 0,
                "compact": int(saved.compact.iat[0]) if len(saved) else 0,
                "kept": len(saved), "secs": 0.0, "reused": True}
    table, found, compact = cluster_group(frame, method, mcs, ms, eps, max_radius)
    table = table.head(keep)
    table.assign(found=found, compact=compact, points=len(frame)).to_csv(path, index=False)
    return {"subset": subset, "slice": slice_name, "points": len(frame), "found": found, "compact": compact,
            "kept": len(table), "secs": round(time.time() - t0, 1), "reused": False}

In [ ]:
jobs = []
for subset, severities in SEVERITIES.items():
    base = df[df.collision_severity.isin(severities)]
    for slice_name in SLICES:
        frame = base if slice_name == "All times" else base[base.time_of_day_bucket == slice_name]
        mcs, ms = PARAMS[(subset, "All times" if slice_name == "All times" else "slice")]
        path = group_file(subset, slice_name, FINAL_METHOD, mcs, ms)
        jobs.append((len(frame), subset, slice_name, frame, mcs, ms, path))
jobs.sort(key=lambda j: -j[0])      # biggest first keeps both CPU cores busy

print(f"Clustering {len(jobs)} groups with {FINAL_METHOD} ({sum(j[0] for j in jobs):,} collisions in total)...")
t0 = time.time()
summary = Parallel(n_jobs=-1, verbose=5)(
    delayed(run_group)(subset, s, frame, FINAL_METHOD, mcs, ms, DBSCAN_EPS, TOP_N, MAX_RADIUS_M, path, FORCE_RERUN)
    for _, subset, s, frame, mcs, ms, path in jobs)
summary = pd.DataFrame(summary)
summary["subset"] = pd.Categorical(summary.subset, list(SEVERITIES), ordered=True)
summary["slice"] = pd.Categorical(summary["slice"], SLICES, ordered=True)
summary = summary.sort_values(["subset", "slice"]).reset_index(drop=True)
print(f"\nFinished in {(time.time() - t0) / 60:.1f} minutes")
print(summary.to_string(index=False))

## 5. Validate and save the hotspot file

The file follows the contract the backend checks: one list of hotspots with `subset`, `slice`, `latitude`, `longitude`, `radius_m`, `collisions`, `fatal`, `serious`, `slight` and `label`. The notebook checks every rule itself before saving. Afterwards, run `tools/validate_hotspots.py` on the downloaded file as a second check.

In [ ]:
rows, next_id = [], 1
for subset in SEVERITIES:
    for slice_name in SLICES:
        mcs, ms = PARAMS[(subset, "All times" if slice_name == "All times" else "slice")]
        saved = pd.read_csv(group_file(subset, slice_name, FINAL_METHOD, mcs, ms))
        for r in saved.itertuples():
            rows.append({
                "id": next_id, "subset": subset, "slice": slice_name,
                "latitude": round(float(r.latitude), 6), "longitude": round(float(r.longitude), 6),
                "radius_m": int(r.radius_m), "collisions": int(r.collisions),
                "fatal": int(r.fatal), "serious": int(r.serious), "slight": int(r.slight),
                "label": None if pd.isna(r.lsoa) else f"LSOA {r.lsoa}",
            })
            next_id += 1

doc = {
    "version": 1,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "method": FINAL_METHOD + (" (DRY RUN, not real results)" if QUICK_TEST_FRACTION else ""),
    "parameters": {
        "distance": "OSGR metres, Euclidean",
        "collisions_used": int(len(df)),
        "severe_means": "Fatal and Serious collisions only",
        "min_cluster_size_all_times_all": PARAMS[("all", "All times")][0],
        "min_samples_all_times_all": PARAMS[("all", "All times")][1],
        "min_cluster_size_slices_all": PARAMS[("all", "slice")][0],
        "min_samples_slices_all": PARAMS[("all", "slice")][1],
        "min_cluster_size_all_times_severe": PARAMS[("severe", "All times")][0],
        "min_samples_all_times_severe": PARAMS[("severe", "All times")][1],
        "min_cluster_size_slices_severe": PARAMS[("severe", "slice")][0],
        "min_samples_slices_severe": PARAMS[("severe", "slice")][1],
        "cluster_selection": "leaf" if FINAL_METHOD == "HDBSCAN" else "not applicable",
        "max_radius_m": MAX_RADIUS_M,
        "hotspot_definition": "a compact cluster: 90% of its collisions within max_radius_m of its centre",
        "tile_km": TILE_M // 1000,
        "tile_overlap_km": MARGIN_M // 1000,
        "top_n_per_group": TOP_N,
        "radius_definition": "90th percentile distance from the centre, at least 25 m",
        "label_definition": "most common recorded LSOA code among the hotspot's collisions (blank when none is recorded)",
        "selection_rule": "highest worst-case silhouette among methods with at most 50% noise in both test squares",
    },
    "subsets": list(SEVERITIES),
    "slices": SLICES,
    "hotspots": rows,
}


def validate(doc):
    problems = []
    for key in ("version", "generated_at", "method", "parameters", "subsets", "slices", "hotspots"):
        if key not in doc:
            problems.append(f"missing key: {key}")
    seen = set()
    for h in doc["hotspots"]:
        tag = f"hotspot {h['id']}"
        if h["id"] in seen:
            problems.append(f"{tag}: duplicate id")
        seen.add(h["id"])
        if h["subset"] not in doc["subsets"]:
            problems.append(f"{tag}: unknown subset {h['subset']}")
        if h["slice"] not in doc["slices"]:
            problems.append(f"{tag}: unknown slice {h['slice']}")
        if not (49 <= h["latitude"] <= 61 and -9 <= h["longitude"] <= 2.5):
            problems.append(f"{tag}: coordinates outside Great Britain")
        if h["radius_m"] <= 0:
            problems.append(f"{tag}: radius must be positive")
        if h["collisions"] != h["fatal"] + h["serious"] + h["slight"]:
            problems.append(f"{tag}: collisions do not equal fatal + serious + slight")
        if h["subset"] == "severe" and h["slight"] != 0:
            problems.append(f"{tag}: a severe hotspot cannot contain Slight collisions")
    return problems


problems = validate(doc)
assert not problems, "Contract problems:\n" + "\n".join(problems[:20])

OUT_FILE = OUT_DIR / ("hotspots_DRY_RUN.json" if QUICK_TEST_FRACTION else "hotspots.json")
OUT_FILE.write_text(json.dumps(doc, separators=(",", ":")), encoding="utf-8")
print(f"Validated {len(rows):,} hotspots. Saved {OUT_FILE.name} ({OUT_FILE.stat().st_size / 1e6:.2f} MB)")

## 6. What the results show

In [ ]:
hot = pd.DataFrame(rows)
by_group = (hot.groupby(["subset", "slice"], observed=True)
            .agg(hotspots_kept=("id", "size"), collisions_in_kept=("collisions", "sum"),
                 fatal_in_kept=("fatal", "sum")).reset_index())
by_group = by_group.merge(summary[["subset", "slice", "points", "found", "compact"]], on=["subset", "slice"])
by_group["share_of_group_in_kept_hotspots_pct"] = (100 * by_group.collisions_in_kept / by_group.points).round(1)
print(by_group.to_string(index=False))

print("\nTop 10 Fatal and Serious hotspots, all times:")
top = hot[(hot.subset == "severe") & (hot.slice == "All times")].sort_values("collisions", ascending=False).head(10)
print(top[["latitude", "longitude", "radius_m", "collisions", "fatal", "serious", "label"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 11), sharex=True, sharey=True)
for ax, slice_name in zip(axes.ravel(), SLICES):
    h = hot[(hot.subset == "severe") & (hot.slice == slice_name)]
    ax.scatter(df.longitude, df.latitude, s=0.05, color="lightgrey", alpha=0.3)
    ax.scatter(h.longitude, h.latitude, s=h.collisions * 4, alpha=0.5, color="crimson", edgecolor="none")
    ax.set_title(f"{slice_name}: {len(h)} severe hotspots")
    ax.set_aspect(1 / np.cos(np.radians(54)))
fig.suptitle("Fatal and Serious hotspots by time of day (dot size = number of collisions)", fontsize=14)
plt.tight_layout()
plt.show()

**Reading the table and maps**
- `hotspots_kept` is capped at `TOP_N` per group. In the run summary above, `found` is every cluster the method found and `compact` is how many were within the radius limit. Hotspots cover only part of the collisions: most collisions are isolated and belong to no hotspot.
- Night and Morning Rush hold few collisions, so their hotspots are fewer and smaller. The minimum cluster size was lowered for every time slice for this reason.
- A hotspot is a statistical concentration of collisions in the past. It does not predict a future crash and does not by itself show a cause.

## 7. Download the results

In [ ]:
to_download = [OUT_FILE, OUT_DIR / "method_comparison.csv"]
try:
    from google.colab import files
    for f in to_download:
        files.download(str(f))
except ImportError:
    print("Not running in Colab. The files are in:", OUT_DIR)
    for f in to_download:
        print(" ", f)

**After downloading**
1. Copy `hotspots.json` to `results/spatial_temporal/hotspots.json` in the repo.
2. Check it against the backend contract: `python tools\validate_hotspots.py results\spatial_temporal\hotspots.json`
3. Restart the backend. `/api/hotspots/meta` should now show the method and the generated time.
4. Keep `method_comparison.csv` for the report.